Incarc noul set de date obtinut

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import re
import pandas as pd

file_path = '/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset_filtered.csv'

df = pd.read_csv(file_path)

print("Numar de randuri:", len(df))
print("Numar de coloane:", len(df.columns))

display(df.head())

In [ ]:
PERS_PER_PAT = {
    "single": 1,
    "double": 2,
    "sofa bed": 1,
    "bunk": 2,
    "futon": 1
}

Corectam max_capacity extras de pe site

In [ ]:
def capacitate_din_paturi(text):
    if not isinstance(text, str):
        return None

    text = text.lower()

    # Eliminam variantele alternative de paturi
    text = re.sub(
        r"\bor\s+\d+\s+(?:large |extra-large )?(?:single|double|sofa bed|bunk|futon)s?",
        "",
        text
    )

    total = 0

    for nr, tip in re.findall(
        r"(\d+)\s+(?:large |extra-large )?(single|double|sofa bed|bunk|futon)",
        text
    ):
        total += int(nr) * PERS_PER_PAT[tip]

    if total > 0:
        return total

    # Daca descrierea indica o camera dubla sau mai multe tipuri de paturi
    if "double" in text or "multiple bed types" in text:
        return 2

    return None


df["capacity"] = df["unit_description"].apply(capacitate_din_paturi)

print("Capacitati lipsa:", df["capacity"].isna().sum())
print(df["capacity"].value_counts().sort_index())

Afisam capacitatile lipsa

In [ ]:
capacitate_lipsa = df[df["capacity"].isna()]

display(
    capacitate_lipsa[
        ["property_id", "name", "property_type", "unit_description",
         "max_capacity", "beds", "bedrooms", "price_per_night"]
    ]
)

Cautam null uri

In [ ]:
null_analysis = pd.DataFrame({
    "Valori lipsa": df.isna().sum(),
    "Procent lipsa (%)": (df.isna().mean() * 100).round(2)
})

null_analysis = null_analysis[null_analysis["Valori lipsa"] > 0]

display(null_analysis.sort_values(by="Valori lipsa", ascending=False))

In [ ]:
df = df.drop(columns=["has_kitchen"])

In [ ]:
df = df.drop(columns=["address"])
df = df.drop(columns=["beds"])
df = df.drop(columns=["stars"])
df = df.drop(columns=["latitude"])
df = df.drop(columns=["longitude"])

stergem randurile fara rating


In [ ]:
df = df.dropna(subset=["num_reviews"])
df = df[df["num_reviews"] > 0].copy()

In [ ]:
df = df.drop(columns=["adults"])
df = df.drop(columns=["max_capacity"])

In [ ]:
df = df.drop(columns=["beds_extracted"])

In [ ]:
display(
    df.groupby("property_type")[["bedrooms", "bathrooms"]]
    .agg(lambda x: x.isna().sum())
)

# Identificam hotelurile
hotel_mask = df["property_type"].str.lower() == "hotel"

# O camera de hotel are automat minim o camera si o baie
df.loc[hotel_mask, "bedrooms"] = df.loc[hotel_mask, "bedrooms"].fillna(1)
df.loc[hotel_mask, "bathrooms"] = df.loc[hotel_mask, "bathrooms"].fillna(1)

remaining_nulls = df.groupby("property_type")[["bedrooms", "bathrooms"]].agg(
    lambda x: x.isna().sum()
)

display(remaining_nulls[remaining_nulls.sum(axis=1) > 0])

Completam la tot ce e cu capacitatea 2 cu o baie si o camera

In [ ]:
mask = df["capacity"] == 2

df.loc[mask, "bedrooms"] = df.loc[mask, "bedrooms"].fillna(1)
df.loc[mask, "bathrooms"] = df.loc[mask, "bathrooms"].fillna(1)

Reparam la apartamente:


In [ ]:
apartments_nulls = df[
    (df["property_type"].str.lower() == "apartment") &
    (df["bedrooms"].isna() | df["bathrooms"].isna())
]

pd.set_option("display.max_colwidth", None)

display(
    apartments_nulls[
        ["property_id", "name", "unit_description",
         "capacity", "bedrooms", "bathrooms", "url", "price_per_night"]
    ]
)

studio = o camera + o baie

In [ ]:
# descrieri care contin "studio"
studio_mask = df["unit_description"].str.contains(
    "studio", case=False, na=False
)

df.loc[studio_mask & df["bedrooms"].isna(), "bedrooms"] = 1
df.loc[studio_mask & df["bathrooms"].isna(), "bathrooms"] = 1

In [ ]:
apartments_nulls = df[
    (df["property_type"].str.lower() == "apartment") &
    (df["bedrooms"].isna() | df["bathrooms"].isna())
]

pd.set_option("display.max_colwidth", None)

display(
    apartments_nulls[
        ["property_id", "name", "unit_description",
         "capacity", "bedrooms", "bathrooms", "url", "price_per_night"]
    ]
)

In [ ]:
guest_house_nulls = df[
    (df["property_type"].str.lower() == "guest house") &
    (df["bedrooms"].isna() | df["bathrooms"].isna())
]

pd.set_option("display.max_colwidth", None)

display(
    guest_house_nulls[
        ["property_id", "name", "unit_description",
         "capacity", "bedrooms", "bathrooms", "url", "price_per_night"]
    ]
)

In [ ]:
# Guest house-urile cu "room" in unit_description si bedrooms lipsa primesc 1
mask = (
    (df["property_type"].str.lower() == "guest house") &
    (df["unit_description"].str.contains("room", case=False, na=False)) &
    (df["bedrooms"].isna())
)

df.loc[mask, "bedrooms"] = 1

In [ ]:
df.loc[1081, "bedrooms"] = 1

In [ ]:
bb_nulls = df[
    (df["property_type"].str.lower() == "bed and breakfast") &
    (df["bedrooms"].isna() | df["bathrooms"].isna())
]

pd.set_option("display.max_colwidth", None)

display(
    bb_nulls[
        ["property_id", "name", "unit_description",
         "capacity", "bedrooms", "bathrooms", "url", "price_per_night"]
    ]
)

In [ ]:
mask = (
    (df["property_type"].str.lower() == "bed and breakfast") &
    (df["bedrooms"].isna())
)

df.loc[mask, "bedrooms"] = 1

print("Valori completate:", mask.sum())
print("Valori lipsa ramase in bedrooms:", df["bedrooms"].isna().sum())


stergem baile

In [ ]:
df = df.drop(columns=["bathrooms"])

vedem daca avem valori anormale la arie

In [ ]:
df.groupby("property_type")["area_m2"].agg(
    total="count",
    valori_lipsa=lambda x: x.isna().sum()
).sort_values("valori_lipsa", ascending=False)

In [ ]:
df = df.drop(columns=["area_m2"])

In [ ]:
null_counts = df.isna().sum()

print("Total valori lipsa in dataset:", null_counts.sum())

display(
    null_counts[null_counts > 0]
    .sort_values(ascending=False)
    .to_frame(name="valori_lipsa")
)

In [ ]:
output_path = "/content/drive/MyDrive/IAPD/Proiect_IAPD/dataset_anomaly.csv"

df.to_csv(output_path, index=False)